## Lab 12

Ian Solberg 

Due March 16, 2026

Professor Piao

In [1]:
# !pip install plotly
# !pip install "nbformat>=4.2.0"
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from statsmodels.tools.eval_measures import rmse
import matplotlib.pyplot as plt

# Step 1: Ingestion from external source
url = 'https://raw.githubusercontent.com/iasolb/reference_data/refs/heads/main/Zillow_ZHVI_2026_Micro.csv'
df = pd.read_csv(url)
df.head()

,Home_Value,Square_Footage,Property_Age,Distance_to_Transit,School_District_Rating
0,329705.74,1941.0,5.5,6.45,Excellent
1,183343.63,1364.3,35.2,2.15,Average
2,354551.73,2386.9,52.4,0.75,Good
3,325773.17,2192.1,50.2,5.25,Excellent
4,359743.12,3069.8,66.5,12.69,Excellent


In [2]:
# Step 2: Defining the formula
# Utilizing the R-style patsy formula interface allows for elegant, readable model specification
formula ='Home_Value ~ Square_Footage + Property_Age + Distance_to_Transit + School_District_Rating'

In [3]:
# Step 3: Fitting the model and printing the summary
model = smf.ols(formula=formula, data=df)
results = model.fit()
print(results.summary())

                            OLS Regression Results                            
Dep. Variable:             Home_Value   R-squared:                       0.766
Model:                            OLS   Adj. R-squared:                  0.765
Method:                 Least Squares   F-statistic:                     542.5
Date:                Wed, 18 Mar 2026   Prob (F-statistic):          2.81e-309
Time:                        14:34:14   Log-Likelihood:                -12072.
No. Observations:                1000   AIC:                         2.416e+04
Df Residuals:                     993   BIC:                         2.419e+04
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
                                          coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------------------
In

In [4]:
# Step 4: Generating predictions
# We extract the predicted values vector to transition from explanation to prediction
y_pred = results.predict(df)
print(y_pred.head())

0    312288.586866
1    223813.440910
2    331610.316284
3    307402.426806
4    392714.851722
dtype: float64


In [5]:
# Step 5: Calculate RMSE between the actuals and the predictions
model_rmse = rmse(df['Home_Value'], y_pred)
print(f"\nThe Predictive RMSE is: ${model_rmse:,.2f}")


The Predictive RMSE is: $42,316.69


In [6]:

import plotly.express as px


# ── CELL: Extract Residuals & Flag Outliers ────────────────────────────────

# ------------------------------------------------------------------
# HOW RESIDUAL EXTRACTION WORKS WITH THE FORMULA API:
#
# When you fit via  smf.ols(formula, data=df).fit(), the results
# object stores two key Series on your original DataFrame index:
#
#   results.fittedvalues  →  ŷ_i = Xβ for each observation i.
#       Equivalent to your y_pred = results.predict(df).
#       These are the model's predicted Home_Values.
#
#   results.resid  →  e_i = y_i − ŷ_i for each observation i.
#       Positive residual = model underpriced the home.
#       Negative residual = model overpriced the home.
#
# The formula API handled one-hot encoding of School_District_Rating
# internally via patsy (C() is implicit for object/string columns),
# using "Average" as the dropped baseline reference category.
# ------------------------------------------------------------------

fitted    = results.fittedvalues   # same values as your y_pred Series
residuals = results.resid          # raw residuals = Home_Value − ŷ

# Compute the standard deviation of the residual vector.
# Points beyond ±2σ are flagged as outliers — under normality,
# roughly 5% of observations should fall outside these bounds.
sigma = np.std(residuals)

# Boolean mask: True where |residual| > 2 × σ
is_outlier = np.abs(residuals) > 2 * sigma

# Categorical label for Plotly's color mapping:
#   "Outlier (>2σ)" → stark crimson    "Inlier" → muted blue
labels = np.where(is_outlier, "Outlier (>2σ)", "Inlier")

# Summary diagnostics (should match your RMSE of ~$42,316.69)
rmse = np.sqrt(np.mean(residuals ** 2))
n_outliers = is_outlier.sum()

print(f"Residual σ       : ${sigma:>12,.2f}")
print(f"±2σ threshold    : ${2 * sigma:>12,.2f}")
print(f"RMSE             : ${rmse:>12,.2f}")
print(f"R²               :  {results.rsquared:>11.4f}")
print(f"Outliers (>±2σ)  :  {n_outliers} / {len(residuals)}  "
      f"({100 * n_outliers / len(residuals):.1f}%)")


# ── CELL: Interactive Plotly Residual Scatter Plot ─────────────────────────

# ------------------------------------------------------------------
# VISUAL LOGIC:
# 1. px.scatter maps fitted values (ŷ) → x-axis,
#    residuals (y − ŷ) → y-axis.
# 2. The 'color' param uses the label array to split outliers
#    from inliers.  color_discrete_map pins exact hex codes:
#       #dc2626 (crimson) for outliers
#       #60a5fa (muted blue) for inliers
# 3. Three horizontal reference lines are overlaid:
#       y =  0   dashed white  — ideal residual mean
#       y = +2σ  dotted crimson — upper outlier boundary
#       y = −2σ  dotted crimson — lower outlier boundary
# 4. Hover shows the observation index, the fitted Home_Value,
#    the residual dollar amount, and how many σ it represents.
# ------------------------------------------------------------------

plot_df = pd.DataFrame({
    "Fitted Home Value (ŷ)": fitted,
    "Residual (y − ŷ)":      residuals,
    "Category":               labels,
    "Obs Index":              fitted.index,
    "|Residual| / σ":         np.abs(residuals) / sigma,
})

fig = px.scatter(
    plot_df,
    x="Fitted Home Value (ŷ)",
    y="Residual (y − ŷ)",
    color="Category",
    color_discrete_map={
        "Outlier (>2σ)": "#dc2626",   # stark crimson for outliers
        "Inlier":        "#60a5fa",   # muted blue for normal points
    },
    hover_data={
        "Obs Index": True,
        "|Residual| / σ": ":.2f",
        "Category": False,
    },
    opacity=0.7,
    title="Residual Forensics — Fitted vs. Residuals  (n=1,000 · R²=0.766)",
)

# ── Zero-line: a well-specified model scatters residuals around y = 0 ──
fig.add_hline(
    y=0,
    line_dash="dash",
    line_color="white",
    opacity=0.3,
    annotation_text="zero line",
    annotation_font_color="gray",
)

# ── ±2σ outlier boundary bands (crimson dotted) ──
fig.add_hline(
    y=2 * sigma,
    line_dash="dot",
    line_color="#dc2626",
    opacity=0.4,
    annotation_text=f"+2σ  (${2 * sigma:,.0f})",
    annotation_position="top left",
    annotation_font_color="#dc2626",
)
fig.add_hline(
    y=-2 * sigma,
    line_dash="dot",
    line_color="#dc2626",
    opacity=0.4,
    annotation_text=f"−2σ  (−${2 * sigma:,.0f})",
    annotation_position="bottom left",
    annotation_font_color="#dc2626",
)

# ── Layout styling ──
fig.update_layout(
    template="plotly_dark",
    paper_bgcolor="#0c0e12",
    plot_bgcolor="#13161c",
    font=dict(family="DM Mono, Courier New, monospace", size=12),
    title_font_size=16,
    legend_title_text="",
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1,
    ),
    xaxis=dict(
        title="Fitted Home Value (ŷ)",
        gridcolor="rgba(255,255,255,0.05)",
        zeroline=False,
        tickprefix="$",
        tickformat=",",
    ),
    yaxis=dict(
        title="Residual  (y − ŷ)",
        gridcolor="rgba(255,255,255,0.05)",
        zeroline=False,
        tickprefix="$",
        tickformat=",",
    ),
    margin=dict(t=80, b=60, l=80, r=30),
    height=560,
)

fig.update_traces(
    marker=dict(size=7, line=dict(width=0.5, color="rgba(0,0,0,0.3)")),
)

fig.show()


# ── CELL: Interpretation Guide ─────────────────────────────────────────────

guide = """
╔══════════════════════════════════════════════════════════════════════╗
║                  HOW TO READ THIS PLOT                             ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                    ║
║  HETEROSCEDASTICITY                                                ║
║  Look for a fan / cone shape.  If the residual spread widens      ║
║  (or narrows) as fitted values increase, variance is not           ║
║  constant → OLS standard errors are unreliable.                    ║
║  • Funnel opening right = error grows with predicted price.        ║
║  • Remedy: log-transform Home_Value, or use WLS / robust SEs.     ║
║                                                                    ║
║  STRUCTURAL BREAKS                                                 ║
║  Watch for a sudden shift in the residual cloud — a cluster        ║
║  that sits well above or below zero at a specific ŷ range.        ║
║  This signals the model coefficients may not hold uniformly.       ║
║  • Your model note says the condition number is large (1.21e+04)   ║
║    — this hints at multicollinearity, which can amplify residual   ║
║    patterns in correlated feature ranges.                          ║
║  • Investigate with a Chow test or add interaction terms           ║
║    (e.g., Square_Footage × School_District_Rating).                ║
║                                                                    ║
║  NON-LINEARITY                                                     ║
║  A curved (U-shaped or arch) pattern means the true relationship   ║
║  is non-linear but the model fits a straight line.                 ║
║  • Your Property_Age coefficient (−814.60) assumes a linear        ║
║    depreciation — try adding Property_Age² as a feature.           ║
║                                                                    ║
║  OUTLIER CLUSTERS (crimson points)                                 ║
║  Isolated crimson dots are expected (~5% under normality).         ║
║  Crimson clusters concentrated in one region suggest a             ║
║  systematic misspecification — the model is missing a feature.     ║
║                                                                    ║
║  IDEAL PATTERN                                                     ║
║  Random scatter with constant width centered on y = 0,             ║
║  no curves, no fans, roughly 5% beyond the ±2σ bands.             ║
║                                                                    ║
╚══════════════════════════════════════════════════════════════════════╝
"""
print(guide)

Residual σ       : $   42,316.69
±2σ threshold    : $   84,633.39
RMSE             : $   42,316.69
R²               :       0.7662
Outliers (>±2σ)  :  37 / 1000  (3.7%)



╔══════════════════════════════════════════════════════════════════════╗
║                  HOW TO READ THIS PLOT                             ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                    ║
║  HETEROSCEDASTICITY                                                ║
║  Look for a fan / cone shape.  If the residual spread widens      ║
║  (or narrows) as fitted values increase, variance is not           ║
║  constant → OLS standard errors are unreliable.                    ║
║  • Funnel opening right = error grows with predicted price.        ║
║  • Remedy: log-transform Home_Value, or use WLS / robust SEs.     ║
║                                                                    ║
║  STRUCTURAL BREAKS                                                 ║
║  Watch for a sudden shift in the residual cloud — a cluster        ║
║  that sits well above or below zero at a specific ŷ range.        ║
║  T